# Playing pokelike from Python

A game runs in a headless browser: there is no window, no internet connection, and
no pixels are ever read. The game's `state` is a JavaScript object that lives in
memory, and the buttons are DOM objects; both exist perfectly well without
anything being drawn.

A notebook is the awkward case for the usual `with` block, because a `with` block
cannot span cells. Starting a run in one cell, taking a move in the next, and
reading the state in the one after is the whole point of using a notebook, so the
`open_game()` function hands back a game that stays alive until you close it.

Run this once first, if you have not already: `uv run pokelike setup`

In [ ]:
import pokelike.interfaces.python.driver as d
print(d.ThreadedGame)

<class 'pokelike.interfaces.python.driver.ThreadedGame'>


## Open a game

Calling `open_game()` starts a local web server for the copy of the game on disk,
then launches a browser pointed at it. The call asks the OS for the port, so two
notebooks never collide.

In [3]:
from pokelike import open_game

game = open_game()
game


ThreadedGame(HostedGame(url='http://127.0.0.1:44667/', watch=False, max_delay=1, scoring=True, load_images=True, seed=None, steps=0))

## Start a run

The seed pins everything: the map, the encounters, the items on offer. Same seed
and same moves give the same run, score included.

Picking the trainer and the starter are the first two decisions of the run, so
the player makes them.

In [4]:
obs = game.reset(seed=42)
obs['screen'], obs['steps'], obs['done']


('trainer-screen', 0, False)

## What a bot sees

The state a bot sees is one dict. The `actions` list is what you choose from, and
its indices are what `step` takes. Running `uv run pokelike schema` prints the
full reference, generated from a live observation.

In [5]:
obs['actions']


[{'kind': 'element',
  'idx': 0,
  'layer': 'trainer-screen',
  'id': 'trainer-boy',
  'label': 'BOY'},
 {'kind': 'element',
  'idx': 1,
  'layer': 'trainer-screen',
  'id': 'trainer-girl',
  'label': 'GIRL'}]

In [6]:
sorted(obs.keys())


['actions',
 'bag',
 'bag_items',
 'can_reorder',
 'done',
 'layer',
 'offered_moves',
 'prompt',
 'run',
 'screen',
 'seed',
 'steps',
 'team',
 'type_items']

## The same thing, readable

The `render` module rebuilds the full display from the state dict. Nothing here
is read off a picture, because the map below is drawn from the nodes and edges
stored in the game's memory.

In [7]:
from pokelike.core import render

print(render.screen(obs))


step 0   screen: trainer-screen   map 0   badges 0

  >> Who are you?

TEAM
  (empty team)

ACTIONS
  [0] BOY
  [1] GIRL


## Take a move

The `step` method takes an index into `obs['actions']` and returns the new state.
Between one decision and the next, the engine does plenty on its own: it plays
out the battle, shows level-ups, and displays banners, and none of that counts as
a choice. As a result, the method only hands control back when there is really
something to decide.

In [8]:
obs = game.step(0)
print(obs['screen'], '|', len(obs['actions']), 'actions')


starter-screen | 3 actions


In [9]:
game.state()

{'layer': 'screen',
 'screen': 'starter-screen',
 'prompt': 'Choose Your Starter!',
 'run': {'run_seed': 2028475091,
  'map': 0,
  'badges': 0,
  'max_team_size': 1,
  'nuzlocke': False,
  'anyone_fainted': False,
  'finished': False,
  'items_this_run': 0,
  'elite': 0},
 'team': [],
 'bag': [],
 'offered_moves': {},
 'type_items': {'Flying': 'sharp_beak',
  'Fire': 'charcoal',
  'Water': 'mystic_water',
  'Electric': 'magnet',
  'Grass': 'miracle_seed',
  'Psychic': 'twisted_spoon',
  'Ground': 'soft_sand',
  'Rock': 'hard_stone',
  'Dragon': 'dragon_fang',
  'Poison': 'poison_barb',
  'Ghost': 'spell_tag',
  'Normal': 'silk_scarf',
  'Steel': 'metal_coat',
  'Dark': 'black_glasses',
  'Fairy': 'pixie_plate',
  'Ice': 'never_melt_ice',
  'Fighting': 'black_belt',
  'Bug': 'silver_powder'},
 'bag_items': [],
 'can_reorder': False,
 'actions': [{'kind': 'element',
   'idx': 0,
   'layer': 'starter-screen',
   'id': None,
   'label': '★ Shiny Bulbasaur Lv. 5 GRASS POISON SP.A 11 SPE 9 H

Run this cell again and again to walk forward one decision at a time.


In [10]:
obs = game.step(0)
print(render.screen(obs))


step 2   screen: map-screen   map 0   badges 0

TEAM
  0. Bulbasaur    Lv 5  ##########  19/19   Grass/Poison  Magical Leaf 40 *

MAP   [here]  <legal move>  x'=done
  layer  0 | [@]
  layer  1 | <o> <x>
  layer  2 |  T   x   T 
  layer  3 |  o   o   i   o 
  layer  4 |  x   T   x 
  layer  5 |  o   M   $   o 
  layer  6 |  T   T   o 
  layer  7 |  x   + 
  layer  8 |  B 

MOVE TUTOR — what each offer replaces
  0. Bulbasaur    Magical Leaf     40   ->  Energy Ball      90  +50

ACTIONS
  [0] go to node n1_0   (catch)
  [1] go to node n1_1   (battle)


In [11]:
game.state()

{'layer': 'screen',
 'screen': 'map-screen',
 'prompt': None,
 'run': {'run_seed': 2028475091,
  'map': 0,
  'badges': 0,
  'max_team_size': 1,
  'nuzlocke': False,
  'anyone_fainted': False,
  'finished': False,
  'items_this_run': 0,
  'elite': 0},
 'team': [{'uid': 1,
   'species_id': 1,
   'name': 'Bulbasaur',
   'level': 5,
   'hp': 19,
   'max_hp': 19,
   'types': ['Grass', 'Poison'],
   'base_stats': {'hp': 45,
    'atk': 49,
    'def': 49,
    'speed': 45,
    'special': 65,
    'spdef': 65},
   'move_tier': 0,
   'item': None,
   'item_id': None,
   'item_desc': None,
   'move': {'name': 'Magical Leaf',
    'power': 40,
    'type': 'Grass',
    'special': True},
   'mega_stone': None,
   'shiny': True}],
 'bag': [],
 'offered_moves': {'0': {'name': 'Energy Ball',
   'power': 90,
   'type': 'Grass',
   'special': True}},
 'type_items': {'Flying': 'sharp_beak',
  'Fire': 'charcoal',
  'Water': 'mystic_water',
  'Electric': 'magnet',
  'Grass': 'miracle_seed',
  'Psychic': 'twist

## The map is a graph

Choosing a node closes every other one on its layer forever, so where a node
leads matters as much as what it is.

In [12]:
print(render.graph_view(obs['map'], colour=True,emoji = True))


  +------------------------
  |          >🏁<
  |          /  \
  |       (🔴)  (👊)
  |       /  \  /  \
  |     🧢    👊    🧢
  |    /  \  /  \  /  \
  |  🔴    🔴    🎁    🔴
  |    \  /  \  /  \  /
  |     👊    🧢    👊
  |    /  \  /  \  /  \
  |  🔴    📖    🔄    🔴
  |    \  /  \  /  \  /
  |     🧢    🧢    🔴
  |       \  /  \  /
  |        👊    💊
  |          \  /
  |           👑
  +------------------------
  >here<  (can go)  .walked.   unseen 


## Team order is a decision, and it is free

Slot 0 leads the next battle. Reordering does not consume the turn, which is why
it is its own verb rather than an entry in `actions`. A full team would otherwise
add fifteen swap pairs beside the real moves at every single map node.

In [13]:
if obs.get('can_reorder'):
    obs = game.reorder(0, 1)
    print(render.team_view(obs['team']))
else:
    print('not enough Pokemon yet')


not enough Pokemon yet


## The score

The score uses the game's own formula.
Compare with `points_no_time` instead, because the time bonus depends on the clock, which is
frozen for reproducibility, so it sits pinned near 1000 and would drown out
everything else.

In [14]:
score = game.score()
score and {k: score[k] for k in ('points', 'points_no_time')}


{'points': 1000, 'points_no_time': 0}

## Play the rest automatically

A bot is a single method that takes the state and returns which action to take.


In [ ]:
from pokelike.bot.base import Bot


class CatchThenFight(Bot):
    """Catch while the team is small; otherwise, take the first option."""

    name = 'demo'

    def act(self, state):
        if len(state.get('team') or []) < 4:
            for i, a in enumerate(state['actions']):
                if a.get('node') == 'catch':
                    return i
        return 0


while not obs.get('done') and obs.get('actions') and game.steps < 200:
    obs = game.step(CatchThenFight().act(obs))

print(obs['screen'], '| badges', (game.last_alive or {}).get('run', {}).get('badges'))

## Whole runs, and comparing bots

The `play` function runs a full game from start to finish and hands back the
decision trace. The `compare` function runs several bots over identical seeds
and pairs them up, because runs vary enormously by luck and two separate
averages mostly measure who drew the nicer maps.

In [ ]:
game.close()   # The compare() function opens its own game.

from pokelike import compare
from pokelike.bot.random_bot import RandomBot

result = compare({'demo': CatchThenFight(), 'random': RandomBot(seed=0)},
                 seeds=range(5), baseline='random')
print(result['table'])

## Close it

Calling `game.close()` stops the browser and the server together. Always run
this cell, because a leaked browser process is what makes the next notebook
launch fail for reasons that have nothing to do with what you changed.

In [17]:
game.close()


RuntimeError: the game thread is gone; open a new game

---

See [the bot framework](../../../../README.md#3-bot-framework-and-competition) for the full
interface, and [the standings](../../../../bots/README.md#standings) if you want
to enter the contest.